# GW170817 PE with SHARPy + `mlgw_bns_jax` — Fixed sky location

Parameter estimation of GW170817 using:
- **`mlgw_bns_jax`**: JAX-based BNS waveform approximant
- **SHARPy** (original): Sequential Monte Carlo sampler — the waveform template is monkey-patched at runtime
- **Cleaned (deglitched) GWOSC data**: C01/v2 strain with the L1 glitch removed

Sky location fixed to the electromagnetic counterpart (NGC 4993):
- $\alpha = 3.44616\;\mathrm{rad}\;(197.45^\circ)$
- $\delta = -0.408084\;\mathrm{rad}\;(-23.38^\circ)$

11 sampled parameters:

| Index | Parameter | Description |
|:---:|---|---|
| 0 | log-distance | $\ln(d_L / \mathrm{Mpc})$ |
| 1 | inclination | $\theta_{JN}$ |
| 2 | $\phi_c$ | coalescence phase |
| 3 | $\psi$ | polarisation |
| 4 | $\mathcal{M}_c$ | chirp mass |
| 5 | $q$ | mass ratio |
| 6 | $t_c$ | coalescence time (relative to trigger) |
| 7 | $\chi_1$ | spin of primary |
| 8 | $\chi_2$ | spin of secondary |
| 9 | $\Lambda_1$ | tidal deformability of primary |
| 10 | $\Lambda_2$ | tidal deformability of secondary |

In [ ]:
from __future__ import annotations

import os, sys, time
from functools import partial

import numpy as np

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

## Load the waveform model and monkey-patch SHARPy

We replace the original `IMRPhenomD` template in SHARPy with our `mlgw_bns_jax` BNS waveform model **without modifying any SHARPy source file**.

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

# ---- Monkey-patch SHARPy's template -----------------------------------
import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses


def _template_mlgw_bns(params, frequency_array):
    """mlgw_bns_jax waveform, drop-in replacement for SHARPy's template."""
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]

    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor


_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
from sharpy.smc_functions import run_sharpy
import sharpy.PSDs

print("Model loaded — SHARPy template patched with mlgw_bns_jax.")

## Event parameters

In [ ]:
TRIGGER_TIME = 1187008882.43
SEGMENT_DURATION = 4.0
SAMPLING_RATE = 4096
F_LOWER = 20.0
F_UPPER = 2000.0
DATA_START_GPS = 1187008867
DATA_DURATION = 32

# Sky location fixed to NGC 4993 (EM counterpart)
FIXED_RA = 3.44616      # rad  (197.45 deg)
FIXED_DEC = -0.408084   # rad  (-23.38 deg)

DATA_DIR = "gw170817_data"
OUTDIR = "outdir_GW170817_sharpy_fixedsky"
LABEL = "GW170817_sharpy_fixedsky"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Sky fixed to NGC 4993: RA = {FIXED_RA:.5f} rad, Dec = {FIXED_DEC:.6f} rad")

## Load cleaned data and build detector network

SHARPy's `load_data` parses the filename to extract GPS start time and duration (`DET-FRAMETYPE-START-DURATION.txt`).
We create symlinks from the cleaned files to LIGO-convention names.

In [ ]:
cleaned_sources = {
    "H1": os.path.join(DATA_DIR, "H1_cleaned.txt"),
    "L1": os.path.join(DATA_DIR, "L1_cleaned.txt"),
    "V1": os.path.join(DATA_DIR, "V1_cleaned.txt"),
}
cleaned_ligo = {
    "H1": os.path.join(DATA_DIR, f"H-H1_CLEANED_C01-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_CLEANED_C01-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_CLEANED_C01-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det in ["H1", "L1", "V1"]:
    src = os.path.abspath(cleaned_sources[det])
    dst = cleaned_ligo[det]
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"{det}: {os.path.basename(dst)} -> {os.path.basename(src)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=cleaned_ligo[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print("\nBuilding GW network...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

## Define likelihood wrapper and priors

11-parameter reduced vector (what the sampler sees):
- `[0]` logdist, `[1]` incl, `[2]` phic, `[3]` pol, `[4]` mc, `[5]` q, `[6]` tc, `[7]` chi1, `[8]` chi2, `[9]` lambda_1, `[10]` lambda_2

Expanded to 13-parameter full vector with RA and Dec fixed (NGC 4993).

In [ ]:
batched_detector = gw_network.batched_detector
log_likelihood_full = partial(log_likelihood_det, detector_list=batched_detector)


def log_likelihood_reduced(params_11):
    """Expand 11 sampled params -> full 13-param SHARPy vector (RA, Dec fixed)."""
    params_13 = jnp.array([
        FIXED_RA,        # [0]  ra      — fixed
        FIXED_DEC,       # [1]  dec     — fixed
        params_11[0],    # [2]  logdistance
        params_11[1],    # [3]  inclination
        params_11[2],    # [4]  phic
        params_11[3],    # [5]  pol
        params_11[4],    # [6]  mc
        params_11[5],    # [7]  q
        params_11[6],    # [8]  tc
        params_11[7],    # [9]  chi1
        params_11[8],    # [10] chi2
        params_11[9],    # [11] lambda_1
        params_11[10],   # [12] lambda_2
    ])
    return log_likelihood_full(params_13)


prior_bounds = jnp.array([
    [jnp.log(10.0), jnp.log(100.0)],  # [0]  logdistance (10–100 Mpc)
    [0.0,           jnp.pi],           # [1]  inclination
    [0.0,           2 * jnp.pi],       # [2]  phic
    [0.0,           jnp.pi],           # [3]  pol
    [1.18,          1.21],             # [4]  mc
    [0.5,           1.0],              # [5]  q
    [-0.1,          0.1],              # [6]  tc
    [-0.05,         0.05],             # [7]  chi1
    [-0.05,         0.05],             # [8]  chi2
    [0.0,           5000.0],           # [9]  lambda_1
    [0.0,           5000.0],           # [10] lambda_2
])

# 1 = periodic, 0 = reflective
boundary_conditions = jnp.array([
    0,  # logdist  (reflective)
    0,  # incl     (reflective)
    1,  # phic     (periodic)
    1,  # pol      (periodic)
    0,  # mc       (reflective)
    0,  # q        (reflective)
    0,  # tc       (reflective)
    0,  # chi1     (reflective)
    0,  # chi2     (reflective)
    0,  # lambda_1 (reflective)
    0,  # lambda_2 (reflective)
])

parameter_names = [
    "logdistance", "theta_jn", "phiref", "pol",
    "mc", "q", "tc", "chi1", "chi2", "lambda_1", "lambda_2",
]


def prior(params):
    """Uniform prior (log-prior = 0 inside bounds)."""
    return 0.0


print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

## Run the SMC sampler

In [ ]:
N_PARTICLES = 500
STEP_SIZE = 0.3
ALPHA = 0.95
SEED = 42

print(f"Starting SHARPy SMC with {N_PARTICLES} particles over {len(parameter_names)} parameters...")
start = time.time()

result_dict = run_sharpy(
    log_likelihood_reduced, prior,
    prior_bounds, boundary_conditions,
    ALPHA, N_PARTICLES, STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR, label=LABEL,
)

dt = time.time() - start
samples = result_dict["posterior_samples"]
logZ, dlogZ = result_dict["logZ"], result_dict["dlogZ"]
print(f"\nDone in {dt:.1f} s — log Z = {logZ:.2f} ± {dlogZ:.2f}")

## Corner plot

In [ ]:
from corner import corner

fig = corner(
    np.array(samples), show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved to {plot_path}")
fig